# Experiment

> This notebook reproduces the results presented in *nanoTabPFN: A Lightweight and Educational Reimplementation of TabPFN*.

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

In [ ]:
import functools
from collections import defaultdict
from collections.abc import Callable, Sequence
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from pydantic import RootModel
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
from tabpfn.preprocessing import PreprocessorConfig

from nanotabpfn import (
    NanoTabPFNModel,
    PriorDumpDataLoader,
    get_default_device,
    set_randomness_seed,
    train,
)
from nanotabpfn.experiment import (
    EvalResults,
    XTypesEnum,
    evaluate_model,
    get_openml_datasets,
    plot_multiple_summary_metrics,
    plot_single_summary_metric,
)


class BaselineResultsFile(RootModel[dict[str, list[EvalResults]]]):
    pass


def apply_aggregation(
    eval_results: list[EvalResults],
    func: Callable[[Sequence[float]], float | np.floating[Any]] = np.mean,
) -> dict[str, Any]:

    results_by_metric = defaultdict(list)

    for result in eval_results:
        for metric, value in result.items():
            results_by_metric[metric].append(value)

    aggregated_results = {metric: func(values) for metric, values in results_by_metric.items()}

    return aggregated_results

## Training

In [ ]:
NUM_SEEDS = 2
# NUM_SEEDS = 20 # If you want to reproduce the paper results, use 20 seeds
DATASETS = get_openml_datasets(max_features=10, new_instances=200, target_classes_filter=2)

In [ ]:
RESULTS_DIR = Path("./experiment-results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BASELINES_PATH = RESULTS_DIR / "baselines.json"
NANO_RUNS_DIR = RESULTS_DIR / "nano-runs"
NANO_RUNS_DIR.mkdir(parents=True, exist_ok=True)

LOAD_BASELINES = True
LOAD_HISTORIES = True
# LOAD_BASELINES = False
# LOAD_HISTORIES = False

### "baselines" - tabpfn and others

In [ ]:
baseline_models = {
    f"TabPFN {ModelVersion.V3_5.value}": [
        TabPFNClassifier.create_default_for_version(ModelVersion.V3_5, random_state=i)
        for i in range(NUM_SEEDS)
    ],
    f"TabPFN {ModelVersion.V3_5.value} (no preprocessing)": [
        TabPFNClassifier.create_default_for_version(
            ModelVersion.V3_5,
            inference_config={
                "PREPROCESS_TRANSFORMS": [PreprocessorConfig(name="none")],
                "FINGERPRINT_FEATURE": False,
            },
            n_estimators=1,
            random_state=i,
        )
        for i in range(NUM_SEEDS)
    ],
    f"TabPFN {ModelVersion.V2.value}": [
        TabPFNClassifier.create_default_for_version(ModelVersion.V2, random_state=i)
        for i in range(NUM_SEEDS)
    ],
    f"TabPFN {ModelVersion.V2.value} (no preprocessing)": [
        TabPFNClassifier.create_default_for_version(
            ModelVersion.V2,
            inference_config={
                "PREPROCESS_TRANSFORMS": [PreprocessorConfig(name="none")],
                "FINGERPRINT_FEATURE": False,
            },
            n_estimators=1,
            random_state=i,
        )
        for i in range(NUM_SEEDS)
    ],
    "Random Forest": [RandomForestClassifier(random_state=i) for i in range(NUM_SEEDS)],
    "Histgradientboost": [HistGradientBoostingClassifier(random_state=i) for i in range(NUM_SEEDS)],
    "K-Nearest Neighbors": [KNeighborsClassifier()],
    "Decision Tree": [DecisionTreeClassifier(random_state=i) for i in range(NUM_SEEDS)],
    "Gaussian Process": [GaussianProcessClassifier(random_state=i) for i in range(NUM_SEEDS)],
}

In [ ]:
if LOAD_BASELINES:
    baseline_results = BaselineResultsFile.model_validate_json(
        BASELINES_PATH.read_text(encoding="utf-8")
    ).root
else:
    baseline_results = {
        name: [evaluate_model(model, datasets=DATASETS) for model in models]
        for name, models in baseline_models.items()
    }

    BASELINES_PATH.write_text(
        BaselineResultsFile(root=baseline_results).model_dump_json(indent=2) + "\n",
        encoding="utf-8",
    )


baselines_mean = pl.from_dicts(
    [
        {"baseline": name, **apply_aggregation(models, np.mean)}
        for name, models in baseline_results.items()
    ]
)
baselines_std = pl.from_dicts(
    [
        {"baseline": name, **apply_aggregation(models, np.std)}
        for name, models in baseline_results.items()
    ]
)

### nanotabpfn

In [ ]:
if LOAD_HISTORIES:
    nano_runs = [
        pl.read_parquet(NANO_RUNS_DIR / f"seed-{seed:03d}.parquet") for seed in range(NUM_SEEDS)
    ]
else:
    device = get_default_device()

    nano_runs = []

    for seed in range(NUM_SEEDS):
        set_randomness_seed(seed)
        prior = PriorDumpDataLoader("300k_150x5_2.h5", num_steps=2500, batch_size=32, device=device)
        model = NanoTabPFNModel(
            embedding_size=96,
            num_attention_heads=4,
            mlp_hidden_size=192,
            num_layers=3,
            num_outputs=2,
        )
        eval_model_partial = functools.partial(evaluate_model, datasets=DATASETS)
        model, history = train(
            model, prior, lr=4e-3, steps_per_eval=25, eval_func=eval_model_partial
        )
        history = [
            {
                **item.scores,
                "training_time": item.training_time,
                "iteration": item.iteration,
            }
            for item in history
        ]

        history = pl.DataFrame(history)
        history.write_parquet(NANO_RUNS_DIR / f"seed-{seed:03d}.parquet")

        nano_runs.append(history)

## Eval

In [ ]:
plot_single_summary_metric(
    nano_runs, baselines=baselines_mean, baselines_std=baselines_std, x_type=XTypesEnum.iteration
)

In [ ]:
with pl.Config(
    set_tbl_formatting="ASCII_MARKDOWN",
    tbl_hide_dtype_separator=True,
    set_tbl_hide_column_data_types=True,
    set_tbl_hide_dataframe_shape=True,
):
    for c in baselines_mean.columns:
        if c.endswith("/ROC AUC"):
            print(f"**{c}**:\n")
            print(baselines_mean.select(["baseline", c]).sort(c, descending=True))
            print("\n")

**Amazon_employee_access/ROC AUC**:

| baseline                       | Amazon_employee_access/ROC AUC |
|--------------------------------|--------------------------------|
| Random Forest                  | 0.635638                       |
| Histgradientboost              | 0.612145                       |
| TabPFN v3.5                    | 0.608821                       |
| K-Nearest Neighbors            | 0.59375                        |
| TabPFN v3.5 (no preprocessing) | 0.581671                       |
| TabPFN v2 (no preprocessing)   | 0.55984                        |
| TabPFN v2                      | 0.533466                       |
| Decision Tree                  | 0.509752                       |
| Gaussian Process               | 0.5                            |


**blood-transfusion-service-center/ROC AUC**:

| baseline                       | blood-transfusion-service-cent... |
|--------------------------------|-----------------------------------|
| TabPFN v3.5 (no preprocessing) | 0.683868                          |
| TabPFN v3.5                    | 0.679825                          |
| TabPFN v2                      | 0.673246                          |
| TabPFN v2 (no preprocessing)   | 0.664611                          |
| Histgradientboost              | 0.602796                          |
| Random Forest                  | 0.578365                          |
| Decision Tree                  | 0.544476                          |
| Gaussian Process               | 0.53639                           |
| K-Nearest Neighbors            | 0.528029                          |


**diabetes/ROC AUC**:

| baseline                       | diabetes/ROC AUC |
|--------------------------------|------------------|
| TabPFN v2                      | 0.837857         |
| TabPFN v2 (no preprocessing)   | 0.83761          |
| TabPFN v3.5                    | 0.832527         |
| TabPFN v3.5 (no preprocessing) | 0.831456         |
| Random Forest                  | 0.821621         |
| Histgradientboost              | 0.80011          |
| K-Nearest Neighbors            | 0.748077         |
| Decision Tree                  | 0.652198         |
| Gaussian Process               | 0.493956         |


In [ ]:
p = plot_multiple_summary_metrics(
    nano_runs, baselines=baselines_mean, baselines_std=baselines_std, x_type=XTypesEnum.iteration
)
# p.save("nanotabpfn_summary.png", dpi=300)
p